# 04 — Full cohort, confirmation fold

CPU. Attach the saved fresh preparation output containing fresh_manifest.csv. Use all eligible studies, with source fold 1 held out for confirmation. The earlier 752-study cache is a screening cohort and is not an input here.

In [ ]:
import importlib.util, importlib.metadata, subprocess, sys
INSTALL_MISSING = True  # enable Kaggle Internet for setup, or attach wheels below
WHEEL_DIR = None
PACKAGES = {'numpy': 'numpy', 'pandas': 'pandas', 'sklearn': 'scikit-learn', 'pydicom': 'pydicom', 'PIL': 'pillow', 'cv2': 'opencv-python-headless'}
missing = [pip for module,pip in PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    if not INSTALL_MISSING: raise RuntimeError(f'Missing dependencies: {missing}. Enable install or attach wheels.')
    cmd = [sys.executable,'-m','pip','install']
    if WHEEL_DIR: cmd += ['--no-index','--find-links',WHEEL_DIR]
    subprocess.check_call(cmd+missing)
print('Dependencies available. Tested MONAI version: 1.5.1; actual:', importlib.metadata.version('monai') if 'monai' in PACKAGES else 'not needed')


In [ ]:
from pathlib import Path
import os, sys, json, importlib, logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s', force=True)
RUNTIME = Path(os.environ.get('RSNA_EXPERIMENT_RUNTIME', '/kaggle/working'))/'rsna_experiments_554732cb8ca9a833'
PACKAGE = RUNTIME/'rsna_experiments'
PACKAGE.mkdir(parents=True, exist_ok=True)
SOURCES = {'__init__.py': '"""Research experiments independent of the historical RSNA pipelines."""\n', 'common.py': '"""Input contracts shared by the standalone experiment notebooks."""\nfrom pathlib import Path\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\n\nUID = \'StudyInstanceUID\'\nTARGETS = [\'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\',\n           \'Lateral OA\', \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s", \'Contusion\', \'Fracture\']\nYC = [f\'y_{i}\' for i in range(12)]\nWC = [f\'w_{i}\' for i in range(12)]\nROOTS = [\'/kaggle/input/datasets/aymenslimani\', \'/kaggle/input\']\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with open(path, \'rb\') as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef fingerprint(obj):\n    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str, allow_nan=False).encode()).hexdigest()\n\n\ndef write_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    temp.write_text(json.dumps(obj, sort_keys=True, indent=2, default=str, allow_nan=False) + \'\\n\', encoding=\'utf-8\')\n    temp.replace(path)\n\n\ndef safe_output(out, identity):\n    out = Path(out)\n    marker = out / \'run_identity.json\'\n    if marker.exists():\n        if json.loads(marker.read_text()) != {\'identity\': identity}:\n            raise ValueError(f\'Output belongs to a different experiment: {out}. Choose a new directory.\')\n    elif out.exists() and any(out.iterdir()):\n        raise ValueError(f\'Unrecognized nonempty output directory: {out}\')\n    out.mkdir(parents=True, exist_ok=True)\n    write_json(marker, {\'identity\': identity})\n    return out\n\n\ndef locate(filename, roots=None, override=None):\n    if override:\n        path = Path(override)\n        if not path.is_file():\n            raise FileNotFoundError(f\'Explicit input does not exist: {path}\')\n        return path\n    roots = ROOTS if roots is None else roots\n    hits = sorted({p.resolve() for root in roots if Path(root).exists()\n                   for p in Path(root).rglob(filename) if p.is_file()})\n    if not hits:\n        raise FileNotFoundError(f\'Attach the saved output containing {filename}; searched {roots}\')\n    if len(hits) > 1 and len({sha256(p) for p in hits}) > 1:\n        raise ValueError(f\'Conflicting {filename} copies; set the explicit path: {hits}\')\n    print(f\'{filename}: {hits[0]}\', flush=True)\n    return hits[0]\n\n\ndef competition_path():\n    for root in [\'/kaggle/input/competitions/rsna-knee-abnormality-detection\',\n                 \'/kaggle/input/rsna-knee-abnormality-detection\']:\n        if (Path(root) / \'train.csv\').is_file():\n            return Path(root)\n    raise FileNotFoundError(\'Attach the RSNA competition dataset.\')\n\n\ndef environment():\n    versions = {}\n    for name in [\'torch\', \'torchvision\', \'monai\', \'timm\', \'pydicom\', \'numpy\', \'pandas\', \'scikit-learn\']:\n        try:\n            versions[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            versions[name] = None\n    return {\'python\': platform.python_version(), \'packages\': versions}\n', 'preparation.py': '"""Freeze an exploratory cohort without changing fresh01 folds or validation labels.\n\n``prepare(config)`` returns the output directory. Required keys: manifest_path,\nout_dir. Optional: preparation_path (defaults to the manifest\'s sibling JSON),\nlabel_audit_path (auto-detected sibling CSV), corrections_path, fold=0,\nmax_train=512, max_val=240, seed=42, audit_sample_per_status=3 (1..20).\nZero cohort limits mean all rows. Manual audit remains required after preparation.\n"""\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nimport re\n\nimport numpy as np\nimport pandas as pd\n\nfrom .common import UID, TARGETS, YC, WC, fingerprint, locate, safe_output, sha256, write_json\n\n\nSCHEMA = "rsna-experiments-preparation-v1"\nIDENTIFIERS = [UID, "group", "fold"]\n\n\ndef _integer(value, name, minimum=0):\n    if isinstance(value, bool):\n        raise ValueError(f"{name} must be an integer >= {minimum}")\n    try:\n        result = int(value)\n        if float(value) != result or result < minimum:\n            raise ValueError\n    except (TypeError, ValueError, OverflowError) as exc:\n        raise ValueError(f"{name} must be an integer >= {minimum}") from exc\n    return result\n\n\ndef _required(frame, columns, context):\n    missing = sorted(set(columns) - set(frame.columns))\n    if missing:\n        raise ValueError(f"{context}: missing columns {missing}")\n\n\ndef _nonempty(frame, columns, context):\n    for col in columns:\n        if frame[col].isna().any() or frame[col].astype(str).str.strip().eq("").any():\n            raise ValueError(f"{context}: null/empty {col}")\n        if not frame[col].astype(str).eq(frame[col].astype(str).str.strip()).all():\n            raise ValueError(f"{context}: surrounding whitespace in {col}")\n\n\ndef _unit_values(frame, columns, context):\n    try:\n        values = frame[columns].apply(pd.to_numeric, errors="raise").to_numpy(dtype=float)\n    except (TypeError, ValueError) as exc:\n        raise ValueError(f"{context}: numeric values required") from exc\n    if not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():\n        raise ValueError(f"{context}: values must be finite and within [0,1]")\n    frame[columns] = values\n\n\ndef _validate_manifest(frame, metadata, fold):\n    _required(frame, IDENTIFIERS + list(YC) + list(WC), "manifest")\n    if frame.empty:\n        raise ValueError("manifest is empty")\n    _nonempty(frame, [UID, "group", "fold"], "manifest")\n    if frame[UID].duplicated().any():\n        raise ValueError("manifest: duplicate UID")\n    try:\n        numeric_folds = pd.to_numeric(frame["fold"], errors="raise")\n    except (TypeError, ValueError) as exc:\n        raise ValueError("source fold must be an integer >= 0") from exc\n    frame["fold"] = [_integer(v, "source fold") for v in numeric_folds]\n    if metadata.get("n_folds") is not None:\n        n_folds = _integer(metadata["n_folds"], "n_folds", 2)\n        if (frame["fold"] >= n_folds).any() or fold >= n_folds:\n            raise ValueError("fold outside source n_folds")\n    if frame.groupby("group")["fold"].nunique().gt(1).any():\n        raise ValueError("grouping overlap: a frozen group occurs in multiple folds")\n    if not frame["fold"].eq(fold).any() or not frame["fold"].ne(fold).any():\n        raise ValueError("requested fold must have both training and validation rows")\n    _unit_values(frame, list(YC) + list(WC), "manifest y/w")\n\n\ndef _select(frame, limit, seed):\n    """Rank identifiers only; labels, row order and report status never select rows."""\n    ranked = frame.copy()\n    ranked["_rank"] = [fingerprint({"seed": seed, UID: u, "group": g, "fold": int(f)})\n                       for u, g, f in ranked[IDENTIFIERS].itertuples(index=False, name=None)]\n    ranked = ranked.sort_values(["_rank", UID])\n    return (ranked.head(limit) if limit else ranked).drop(columns="_rank")\n\n\ndef _counts(frame):\n    rows = []\n    for fold, subset in frame.groupby("fold", sort=True):\n        for target, ycol, wcol in zip(TARGETS, YC, WC):\n            known = subset[wcol].gt(0)\n            rows.append({"fold": int(fold), "target": target, "n": len(subset),\n                         "positive": int((known & subset[ycol].eq(1)).sum()),\n                         "negative": int((known & subset[ycol].eq(0)).sum()),\n                         "unknown": int((~known).sum()),\n                         "soft": int((known & subset[ycol].between(0, 1, inclusive="neither")).sum()),\n                         "weight_sum": float(subset[wcol].sum())})\n    return rows\n\n\ndef _prior_comparison(frame, fold):\n    """Training-only weighted priors; undefined targets remain explicit nulls."""\n    train = frame.loc[frame["fold"].ne(fold)]\n    result = []\n    for target, ycol, wcol in zip(TARGETS, YC, WC):\n        weight = float(train[wcol].sum())\n        prior = float((train[ycol] * train[wcol]).sum() / weight) if weight else None\n        row = {"target": target, "training_weight": weight, "training_prior": prior}\n        for name, mask in (("train", frame["fold"].ne(fold)), ("validation", frame["fold"].eq(fold))):\n            subset = frame.loc[mask]\n            denom = float(subset[wcol].sum())\n            if prior is None or not denom:\n                bce = None\n            else:\n                p = np.clip(prior, 1e-7, 1 - 1e-7)\n                losses = -subset[ycol] * np.log(p) - (1 - subset[ycol]) * np.log1p(-p)\n                bce = float((losses * subset[wcol]).sum() / denom)\n            row[f"{name}_bce"] = bce\n        result.append(row)\n    return {"purpose": "constant-prediction diagnostic, no scoring claim",\n            "prior_scope": "selected training rows after explicit corrections",\n            "bce": "per-target sum(weight * soft BCE) / sum(weight)",\n            "clip_epsilon": 1e-7, "targets": result}\n\n\ndef _audit(path, source, selected, fold, seed, sample_limit):\n    columns = [UID, "group", "fold", "target", "status", "source_status", "label", "weight",\n               "report", "report_truncated", "manual_review_required"]\n    audit = None\n    summary = {"available": path is not None, "cross_group_collision_count": 0,\n               "cross_group_collisions": [], "report_normalization": "lowercase, collapse whitespace, strip",\n               "manual_review_required": True}\n    if path is not None:\n        audit = pd.read_csv(path, dtype=str, keep_default_na=False)\n        _required(audit, [UID, "report"] + list(TARGETS), "label audit")\n        _nonempty(audit, [UID], "label audit")\n        if audit[UID].duplicated().any():\n            raise ValueError("label audit: duplicate UID")\n        statuses = {"positive", "negative", "uncertain", "unmentioned", "conflict"}\n        if not audit[list(TARGETS)].isin(statuses).all().all():\n            raise ValueError("label audit: invalid target status")\n        missing = set(source[UID]) - set(audit[UID])\n        if missing:\n            raise ValueError(f"label audit: missing {len(missing)} source UIDs")\n        joined = source[IDENTIFIERS].merge(audit[[UID, "report"]], on=UID, validate="one_to_one")\n        joined["_normalized"] = joined["report"].map(lambda s: re.sub(r"\\s+", " ", s.lower()).strip())\n        summary["empty_report_count"] = int(joined["_normalized"].eq("").sum())\n        # Empty reports contain no evidence and must not imply shared patient identity.\n        collisions = []\n        for report, group in joined.loc[joined["_normalized"].ne("")].groupby("_normalized", sort=True):\n            if group["group"].nunique() <= 1:\n                continue\n            if group["fold"].eq(fold).any() and group["fold"].ne(fold).any():\n                raise ValueError("normalized report collision across training/validation groups")\n            collisions.append({"report_fingerprint": fingerprint(report), "study_count": len(group),\n                               "group_count": int(group["group"].nunique()),\n                               "folds": sorted(int(f) for f in group["fold"].unique())})\n        summary["cross_group_collision_count"] = len(collisions)\n        summary["cross_group_collisions"] = collisions\n        audit = audit.set_index(UID)\n    samples = []\n    ordered = _select(selected, 0, seed)\n    for target, ycol, wcol in zip(TARGETS, YC, WC):\n        used = {}\n        for record in ordered.to_dict("records"):\n            y, w, uid = record[ycol], record[wcol], record[UID]\n            status = "unknown" if w == 0 else "positive" if y == 1 else "negative" if y == 0 else "soft"\n            source_status = str(audit.at[uid, target]) if audit is not None else "unavailable"\n            key = (status, source_status)\n            if used.get(key, 0) >= sample_limit:\n                continue\n            used[key] = used.get(key, 0) + 1\n            report = str(audit.at[uid, "report"]) if audit is not None else ""\n            samples.append({UID: uid, "group": record["group"], "fold": int(record["fold"]),\n                            "target": target, "status": status, "source_status": source_status,\n                            "label": y, "weight": w, "report": report[:2000],\n                            "report_truncated": len(report) > 2000, "manual_review_required": True})\n    summary["sample_rows"] = len(samples)\n    summary["sample_per_target_status"] = sample_limit\n    summary["sample_report_character_limit"] = 2000\n    return summary, pd.DataFrame(samples, columns=columns)\n\n\ndef _correct(frame, source, path, fold):\n    if path is None:\n        return frame, []\n    corrections = pd.read_csv(path, dtype=str, keep_default_na=False)\n    fields = [UID, "target", "label", "weight", "reviewer", "reason"]\n    _required(corrections, fields, "corrections")\n    _nonempty(corrections, [UID, "target", "reviewer", "reason"], "corrections")\n    if corrections.duplicated([UID, "target"]).any():\n        raise ValueError("corrections: duplicate UID/target")\n    _unit_values(corrections, ["label", "weight"], "corrections")\n    if not set(corrections["target"]).issubset(TARGETS):\n        raise ValueError("corrections: invalid target")\n    if not set(corrections[UID]).issubset(set(source[UID])):\n        raise ValueError("corrections: invalid UID")\n    if set(corrections[UID]) & set(source.loc[source["fold"].eq(fold), UID]):\n        raise ValueError("corrections: validation edits are forbidden")\n    if not set(corrections[UID]).issubset(set(frame[UID])):\n        raise ValueError("corrections: UID outside selected training cohort")\n    indexed = frame.set_index(UID).copy()\n    changes = []\n    for item in corrections[fields].sort_values([UID, "target"]).to_dict("records"):\n        i = list(TARGETS).index(item["target"])\n        uid = item[UID]\n        changes.append({**item, "previous_label": float(indexed.at[uid, YC[i]]),\n                        "previous_weight": float(indexed.at[uid, WC[i]])})\n        indexed.at[uid, YC[i]] = item["label"]\n        indexed.at[uid, WC[i]] = item["weight"]\n    return indexed.reset_index()[frame.columns], changes\n\n\ndef prepare(config) -> Path:\n    """Validate fresh01 provenance and write a fixed, auditable experiment cohort."""\n    if not config.get("manifest_path") or not config.get("out_dir"):\n        raise ValueError("manifest_path and out_dir are required")\n    fold = _integer(config.get("fold", 0), "fold")\n    seed = _integer(config.get("seed", 42), "seed")\n    max_train = _integer(config.get("max_train", 512), "max_train")\n    max_val = _integer(config.get("max_val", 240), "max_val")\n    sample_limit = _integer(config.get("audit_sample_per_status", 3), "audit_sample_per_status", 1)\n    if sample_limit > 20:\n        raise ValueError("audit_sample_per_status must be <= 20")\n    manifest_path = Path(locate("fresh_manifest.csv", override=config["manifest_path"])).resolve()\n    metadata_path = Path(config.get("preparation_path") or manifest_path.with_name("fresh_preparation.json")).resolve()\n    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))\n    if not isinstance(metadata, dict) or metadata.get("targets") != list(TARGETS):\n        raise ValueError("fresh preparation targets/order do not match common.TARGETS")\n    source_hash = sha256(manifest_path)\n    if metadata.get("manifest_sha256") != source_hash:\n        raise ValueError("source manifest SHA256 does not match fresh_preparation metadata")\n    source = pd.read_csv(manifest_path, dtype=str, keep_default_na=False)\n    _validate_manifest(source, metadata, fold)\n    # Allowlist: arbitrary input columns (especially report text) never enter training.\n    source = source[IDENTIFIERS + list(YC) + list(WC)].copy()\n    selected = pd.concat([_select(source.loc[source["fold"].ne(fold)], max_train, seed),\n                          _select(source.loc[source["fold"].eq(fold)], max_val, seed)])\n    selected = selected.sort_values(IDENTIFIERS).reset_index(drop=True)\n    cohort_id = fingerprint(selected[IDENTIFIERS].to_dict("records"))\n    corrections_path = Path(config["corrections_path"]).resolve() if config.get("corrections_path") else None\n    selected, changes = _correct(selected, source, corrections_path, fold)\n    validation = source.set_index(UID).loc[selected.loc[selected["fold"].eq(fold), UID], list(YC) + list(WC)]\n    if not validation.equals(selected.loc[selected["fold"].eq(fold)].set_index(UID)[list(YC) + list(WC)]):\n        raise ValueError("validation supervision changed")\n    supervision_id = fingerprint(selected[[UID] + list(YC) + list(WC)].to_dict("records"))\n    audit_override = config.get("label_audit_path") or config.get("audit_path")\n    audit_path = Path(audit_override).resolve() if audit_override else manifest_path.with_name("label_audit.csv")\n    if not audit_override and not audit_path.exists():\n        audit_path = None\n    audit_summary, samples = _audit(audit_path, source, selected, fold, seed, sample_limit)\n    grouping_method = metadata.get("group_method", metadata.get("grouping_method", "unknown"))\n    if not isinstance(grouping_method, str) or not grouping_method.strip():\n        grouping_method = "unknown"\n    patient_evidence = grouping_method in {"PatientID", "patient_id", "PatientId"}\n    provenance = {"manifest": {"path": str(manifest_path), "sha256": source_hash},\n                  "preparation": {"path": str(metadata_path), "sha256": sha256(metadata_path)},\n                  "label_audit": {"path": str(audit_path), "sha256": sha256(audit_path)} if audit_path else None,\n                  "corrections": {"path": str(corrections_path), "sha256": sha256(corrections_path)} if corrections_path else None,\n                  "manifest_hash_verified": True}\n    identity = {"schema": SCHEMA, "cohort_id": cohort_id, "supervision_id": supervision_id,\n                "fold": fold, "seed": seed, "max_train": max_train, "max_val": max_val,\n                "audit_sample_per_status": sample_limit, "source_provenance": provenance}\n    out = Path(config["out_dir"]).resolve()\n    # Never overwrite source inputs, even if safe_output permits an identity reuse.\n    outputs = {out / name for name in ("experiment_manifest.csv", "preparation_meta.json", "run_identity.json", "target_status_audit_sample.csv")}\n    if outputs & {p for p in (manifest_path, metadata_path, audit_path, corrections_path) if p is not None}:\n        raise ValueError("output would overwrite a source input")\n    safe_output(out, identity)\n    output_manifest = out / "experiment_manifest.csv"\n    selected.to_csv(output_manifest, index=False)\n    samples.to_csv(out / "target_status_audit_sample.csv", index=False)\n    result = {"schema": SCHEMA, "cohort_id": cohort_id, "supervision_id": supervision_id,\n              "fold": fold, "manifest_sha256": sha256(output_manifest), "grouping_method": grouping_method,\n              "targets": list(TARGETS), "source_provenance": provenance,\n              "patient_grouping": "source_patient_identifier_unverified" if patient_evidence else "unknown",\n              "exploratory": True, "readiness": "manual_review_required", "manual_review_required": True,\n              "source_rows": len(source), "train_rows": int(selected["fold"].ne(fold).sum()),\n              "validation_rows": int(selected["fold"].eq(fold).sum()),\n              "selection": {"method": "seeded identifier fingerprint rank; no label stratification",\n                            "seed": seed, "max_train": max_train, "max_val": max_val},\n              "source_target_fold_counts": _counts(source), "target_fold_counts": _counts(selected),\n              "constant_prediction_comparison": _prior_comparison(selected, fold),\n              "audit": audit_summary, "corrections": changes, "validation_labels_preserved": True,\n              "warnings": ["Heuristic supervision requires manual target/status review."]}\n    if not patient_evidence:\n        result["warnings"].append("Patient grouping unknown: exploratory only; report grouping does not establish patient independence.")\n    if audit_path is None:\n        result["warnings"].append("No report audit supplied; normalized-report leakage audit unavailable.")\n    write_json(out / "preparation_meta.json", result)\n    return out\n', 'preprocessing.py': '"""Bounded, label-independent MRI slice stacks; no canonical/isotropic claim.\n\nPlane order is SAG, COR, AX. Depth sampling is ordinal after geometric ordering;\npadding is zero and invalid. Only selected instances are pixel-decoded. Source\nSHA256 auditing reads compressed bytes, including unselected instances. Recipes\nchange exactly one baseline operation. Percentiles/z-score use all pixels of the\nselected stack, not the entire acquisition (which would require extra decoding).\n"""\nfrom pathlib import Path\nimport csv\nimport json\nimport logging\nimport math\nimport platform\nimport re\nimport shutil\nimport time\nimport zipfile\n\nimport numpy as np\nimport pydicom\nfrom PIL import Image\n\nfrom .common import UID, fingerprint, locate, sha256, write_json\n\nLOG = logging.getLogger(__name__)\nSCHEMA = "rsna-experiments-cache-v1"\nVERSION = "bounded-mri-1"\nPLANES = ("SAG", "COR", "AX")\nRECIPES = {\n    "p0": dict(band=(.05, .95), crop_mm=None, normalization="percentile"),\n    "p1": dict(band=(.05, .95), crop_mm=130., normalization="percentile"),\n    "p2": dict(band=(.05, .95), crop_mm=150., normalization="percentile"),\n    "p3": dict(band=(.05, .95), crop_mm=None, normalization="zscore"),\n    "p4": dict(band=(.05, .95), crop_mm=None, normalization="clahe"),\n    "p5": dict(band=(.12, .88), crop_mm=None, normalization="percentile"),\n}\n\n\ndef _read_csv(path):\n    with Path(path).open(encoding="utf-8-sig", newline="") as stream:\n        reader = csv.DictReader(stream)\n        return list(reader), reader.fieldnames or []\n\n\ndef _csv(path, rows, fields):\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    with temporary.open("w", encoding="utf-8", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows(rows)\n    temporary.replace(path)\n\n\ndef _component(value):\n    if not re.fullmatch(r"[A-Za-z0-9_.-]+", value or "") or value in {".", ".."}:\n        raise ValueError("Unsafe or empty study/series identifier")\n    return value\n\n\ndef _vector(ds, key, length):\n    try:\n        value = np.asarray(getattr(ds, key), dtype=np.float64)\n    except (AttributeError, TypeError, ValueError) as exc:\n        raise ValueError("invalid_" + key) from exc\n    if value.shape != (length,) or not np.isfinite(value).all():\n        raise ValueError("invalid_" + key)\n    return value\n\n\ndef _axes(ds):\n    values = _vector(ds, "ImageOrientationPatient", 6).reshape(2, 3)\n    if not np.allclose(np.linalg.norm(values, axis=1), 1, atol=.001) or abs(values[0] @ values[1]) > .001:\n        raise ValueError("invalid_orientation_axes")\n    values = values / np.linalg.norm(values, axis=1)[:, None]\n    return values, np.cross(values[0], values[1])\n\n\ndef _semantics(row, ds):\n    """Explicit metadata only: absence of FS never implies T1."""\n    description = " ".join([str(row.get("SeriesDescription", "")), str(getattr(ds, "SeriesDescription", ""))]).lower()\n    raw_type = getattr(ds, "ImageType", [])\n    image_type = raw_type if isinstance(raw_type, str) else " ".join(str(x) for x in raw_type)\n    if re.search(r"locali[sz]er|scout|survey|\\bloc\\b", description + " " + image_type, re.I):\n        raise ValueError("localizer_excluded")\n    aliases = {"sag": "SAG", "sagittal": "SAG", "cor": "COR", "coronal": "COR", "ax": "AX", "axial": "AX", "transverse": "AX", "tra": "AX"}\n    explicit = aliases.get(str(row.get("Anatomical_Plane", "")).strip().lower())\n    tokens = set(re.findall(r"[a-z0-9]+", description))\n    hints = {aliases[t] for t in tokens if t in aliases}\n    if len(hints) > 1 or (explicit and hints and explicit not in hints):\n        raise ValueError("conflicting_plane_metadata")\n    _, normal = _axes(ds)\n    axis = int(np.argmax(abs(normal)))\n    if abs(normal[axis]) < math.cos(math.radians(30)):\n        raise ValueError("ambiguous_oblique_plane")\n    geometric = ("SAG", "COR", "AX")[axis]\n    plane = explicit or next(iter(hints), None) or geometric\n    if plane != geometric:\n        raise ValueError("plane_metadata_geometry_conflict")\n    non_fs = bool(re.search(r"non[ _-]?(?:fs|fat[ _-]?sat)|without\\s+fat", description))\n    fat = str(row.get("Fat_Suppression", "")).strip().lower()\n    fs = fat in {"1", "1.0", "true", "yes", "fs"} or (not non_fs and bool(tokens & {"fs", "fatsat", "stir", "spair"} or re.search(r"fat[ _-]?(?:sat|supp)", description)))\n    if fat in {"0", "0.0", "false", "no"} or non_fs:\n        fs = False\n    fluid = bool(tokens & {"t2", "t2w", "pd", "pdw", "stir"})\n    return plane, (int(fluid and fs), int(fs), int(fluid))\n\n\ndef _headers(folder, row, uid, series, recipe):\n    records = []\n    files = sorted(p for p in folder.iterdir() if p.is_file())\n    if not files:\n        raise ValueError("empty_series")\n    for path in files:\n        ds = pydicom.dcmread(path, stop_before_pixels=True)\n        axes, normal = _axes(ds)\n        position = _vector(ds, "ImagePositionPatient", 3)\n        if int(getattr(ds, "NumberOfFrames", 1)) != 1:\n            raise ValueError("multiframe_rejected")\n        if str(getattr(ds, "StudyInstanceUID", "")) != uid or str(getattr(ds, "SeriesInstanceUID", "")) != series:\n            raise ValueError("source_uid_mismatch")\n        if int(getattr(ds, "SamplesPerPixel", 0)) != 1 or str(getattr(ds, "PhotometricInterpretation", "")) not in {"MONOCHROME1", "MONOCHROME2"}:\n            raise ValueError("unsupported_photometric")\n        shape = (int(ds.Rows), int(ds.Columns))\n        if min(shape) < 1 or max(shape) > 4096 or shape[0] * shape[1] > 4096**2:\n            raise ValueError("source_dimensions_out_of_bounds")\n        spacing = _vector(ds, "PixelSpacing", 2) if hasattr(ds, "PixelSpacing") or recipe["crop_mm"] else None\n        if spacing is not None and (spacing <= 0).any():\n            raise ValueError("invalid_PixelSpacing")\n        plane, preference = _semantics(row, ds)\n        records.append(dict(path=path, ds=ds, axes=axes, normal=normal, ipp=position,\n                            shape=shape, spacing=spacing, plane=plane, preference=preference))\n    first = records[0]\n    sop = set()\n    for item in records:\n        if not np.allclose(item["axes"], first["axes"], atol=1e-4, rtol=0):\n            raise ValueError("inconsistent_orientation")\n        if item["shape"] != first["shape"] or item["plane"] != first["plane"]:\n            raise ValueError("inconsistent_dimensions_or_plane")\n        a, b = item["spacing"], first["spacing"]\n        if (a is None) != (b is None) or (a is not None and not np.allclose(a, b, atol=1e-4, rtol=0)):\n            raise ValueError("inconsistent_PixelSpacing")\n        if str(getattr(item["ds"], "FrameOfReferenceUID", "")) != str(getattr(first["ds"], "FrameOfReferenceUID", "")):\n            raise ValueError("mixed_frame_of_reference")\n        instance = str(getattr(item["ds"], "SOPInstanceUID", ""))\n        if not instance or instance in sop:\n            raise ValueError("missing_or_duplicate_sop")\n        sop.add(instance)\n        offset = item["ipp"] - first["ipp"]\n        if np.linalg.norm(offset - (offset @ first["normal"]) * first["normal"]) > .1:\n            raise ValueError("inconsistent_inplane_origins")\n        item["position"] = float(item["ipp"] @ first["normal"])\n    records.sort(key=lambda item: item["position"])\n    gaps = np.diff([item["position"] for item in records])\n    if len(gaps) and (gaps <= 1e-3).any():\n        raise ValueError("duplicate_slice_positions")\n    # Missing/nonuniform slice intervals are rejected, not interpolated silently.\n    if len(gaps) > 1 and not np.allclose(gaps, np.median(gaps), rtol=.1, atol=.1):\n        raise ValueError("nonuniform_slice_spacing")\n    if recipe["crop_mm"]:\n        wanted = np.rint(recipe["crop_mm"] / first["spacing"]).astype(int)\n        if (wanted < 1).any() or (wanted > first["shape"]).any():\n            raise ValueError("physical_crop_exceeds_fov")\n    return records\n\n\ndef _series_stack(records, cfg):\n    recipe = RECIPES[cfg["recipe"]]\n    low, high = recipe["band"]\n    # Slice-center quantiles give meaningful bands even for short acquisitions.\n    centers = (np.arange(len(records)) + .5) / len(records)\n    candidates = np.flatnonzero((centers >= low) & (centers <= high))\n    selected = candidates[np.rint(np.linspace(0, len(candidates)-1, min(cfg["slices"], len(candidates)))).astype(int)]\n    if len(selected) * math.prod(records[0]["shape"]) > 32 * 1024**2:\n        raise ValueError("selected_source_stack_exceeds_memory_bound")\n    decoded = []\n    for index in selected:\n        record = records[int(index)]\n        ds = pydicom.dcmread(record["path"])\n        pixels = np.asarray(ds.pixel_array, dtype=np.float32)\n        if pixels.shape != record["shape"]:\n            raise ValueError("decoded_shape_mismatch")\n        slope, intercept = float(getattr(ds, "RescaleSlope", 1)), float(getattr(ds, "RescaleIntercept", 0))\n        if not math.isfinite(slope) or slope == 0 or not math.isfinite(intercept):\n            raise ValueError("invalid_rescale")\n        pixels = pixels * slope + intercept\n        if not np.isfinite(pixels).all():\n            raise ValueError("nonfinite_pixels")\n        if ds.PhotometricInterpretation == "MONOCHROME1":\n            pixels = pixels.min() + pixels.max() - pixels\n        decoded.append(pixels)\n    stack = np.stack(decoded)\n    # Normalize before crop so physical-crop recipes change only the FOV.\n    if recipe["normalization"] == "zscore":\n        deviation = float(stack.std(dtype=np.float64))\n        stack = np.full_like(stack, .5) if deviation < 1e-6 else np.clip((stack - stack.mean(dtype=np.float64)) / deviation, -3, 3) / 6 + .5\n    else:\n        lower, upper = np.percentile(stack, [1, 99])\n        stack = np.zeros_like(stack) if upper - lower < 1e-6 else np.clip((stack - lower) / (upper - lower), 0, 1)\n    stack = np.rint(stack * 255).astype(np.uint8)\n    if recipe["crop_mm"]:\n        h, w = np.rint(recipe["crop_mm"] / records[0]["spacing"]).astype(int)\n        top, left = (stack.shape[1] - h) // 2, (stack.shape[2] - w) // 2\n        stack = stack[:, top:top+h, left:left+w]\n    result = np.zeros((cfg["slices"], cfg["size"], cfg["size"]), np.uint8)\n    mask = np.zeros(cfg["slices"], bool)\n    clahe = None\n    if recipe["normalization"] == "clahe":\n        import cv2\n        clahe = cv2.createCLAHE(clipLimit=2., tileGridSize=(8, 8))\n    for i, pixels in enumerate(stack):\n        pixels = np.asarray(Image.fromarray(pixels).resize((cfg["size"], cfg["size"]), Image.Resampling.BOX))\n        result[i] = clahe.apply(pixels) if clahe is not None else pixels\n        mask[i] = True\n    return result, mask, [records[int(i)]["position"] for i in selected]\n\n\ndef _folders(comp, uid):\n    studies = [comp / root / uid for root in ("train_series", "train_images")]\n    hits = [path for path in studies if path.is_dir()]\n    if len(hits) > 1:\n        raise ValueError("ambiguous_study_directories")\n    if not hits:\n        return []\n    if not hits[0].resolve().is_relative_to(comp):\n        raise ValueError("source_path_escapes_competition")\n    result = sorted(p for p in hits[0].iterdir() if p.is_dir())\n    for path in result:\n        _component(path.name)\n        if not path.resolve().is_relative_to(comp):\n            raise ValueError("source_path_escapes_competition")\n    return result\n\n\ndef _source_hash(folders, comp):\n    records = []\n    for folder in folders:\n        records.append([folder.relative_to(comp).as_posix(), "directory"])\n        for path in sorted(folder.iterdir()):\n            if path.is_file():\n                if not path.resolve().is_relative_to(comp):\n                    raise ValueError("source_path_escapes_competition")\n                records.append([path.relative_to(comp).as_posix(), sha256(path)])\n    return fingerprint(records)\n\n\ndef _reuse(path, sidecar, image_id, source_hash, cfg):\n    try:\n        info = json.loads(sidecar.read_text(encoding="utf-8"))\n        if info["image_id"] != image_id or info["source_hash"] != source_hash or not info["success"] or sha256(path) != info["sha256"]:\n            return None\n        with np.load(path, allow_pickle=False) as archive:\n            images, mask = archive["images"], archive["mask"]\n            if images.dtype != np.uint8 or images.shape != (3, cfg["slices"], cfg["size"], cfg["size"]) or mask.dtype != bool or mask.shape != (3, cfg["slices"]):\n                return None\n            if not mask.any() or np.any(images[~mask]) or int(mask.sum()) != info["valid_slices"]:\n                return None\n        return info\n    except (OSError, ValueError, KeyError, TypeError, EOFError, zipfile.BadZipFile):\n        return None\n\n\ndef _study(folders, metadata, uid, cfg):\n    images = np.zeros((3, cfg["slices"], cfg["size"], cfg["size"]), np.uint8)\n    mask = np.zeros((3, cfg["slices"]), bool)\n    choices, errors, chosen = {}, [], {}\n    for folder in folders:\n        try:\n            records = _headers(folder, metadata.get((uid, folder.name), {}), uid, folder.name, RECIPES[cfg["recipe"]])\n            first = records[0]\n            rank = tuple(-n for n in first["preference"]) + (-len(records), folder.name)\n            choices.setdefault(first["plane"], []).append((rank, folder.name, records))\n        except Exception as exc:\n            errors.append(dict(series=folder.name, stage="header", error=f"{type(exc).__name__}: {exc}"))\n    for slot, plane in enumerate(PLANES):\n        if plane not in choices:\n            continue\n        _, series, records = min(choices[plane], key=lambda item: item[0])\n        chosen[plane] = dict(series=series, positions=[], success=False)\n        try:\n            stack, valid, positions = _series_stack(records, cfg)\n            images[slot], mask[slot] = stack, valid\n            chosen[plane].update(positions=positions, success=True)\n        except Exception as exc:\n            errors.append(dict(series=series, stage="decode", error=f"{type(exc).__name__}: {exc}"))\n    if not mask.any():\n        errors.append(dict(series="", stage="study", error="no_usable_series"))\n    return images, mask, dict(success=bool(mask.any()), valid_slices=int(mask.sum()), selected=chosen, errors=errors)\n\n\ndef build_cache(config) -> Path:\n    """Build ``out_dir/recipe`` and return that directory, including on failure.\n\n    Check cache_meta.json complete before training. All manifest rows survive;\n    max_studies processes the first N rows and leaves explicit unprocessed index\n    entries, making a truncated run incomplete. Relabeling identical study rows\n    refreshes provenance without touching valid label-free NPZs. Different image\n    configuration/cohort requires a new output directory. Single writer only.\n    """\n    started = time.monotonic()\n    cfg = {key: config.get(key, default) for key, default in dict(recipe="p0", slices=16, size=192, max_studies=0).items()}\n    if cfg["recipe"] not in RECIPES:\n        raise ValueError("recipe must be p0 through p5")\n    for key in ("slices", "size", "max_studies"):\n        if isinstance(cfg[key], bool) or not isinstance(cfg[key], (int, np.integer)) or cfg[key] < (0 if key == "max_studies" else 1):\n            raise ValueError(f"{key} must be a {\'nonnegative\' if key == \'max_studies\' else \'positive\'} integer")\n        cfg[key] = int(cfg[key])\n    if cfg["slices"] > 256 or cfg["size"] > 1024:\n        raise ValueError("Bounded cache requires slices <= 256 and size <= 1024")\n    comp = Path(config["comp_dir"]).resolve(strict=True)\n    manifest = Path(locate("experiment_manifest.csv", roots=config.get("input_roots"), override=config.get("manifest_path"))).resolve()\n    preparation_path = manifest.with_name("preparation_meta.json")\n    preparation = json.loads(preparation_path.read_text(encoding="utf-8-sig"))\n    if preparation.get("complete") is False:\n        raise ValueError("Preparation is incomplete")\n    for key in ("cohort_id", "supervision_id"):\n        if not preparation.get(key):\n            raise ValueError("Preparation missing " + key)\n    fold = preparation.get("fold", preparation.get("config", {}).get("fold"))\n    if isinstance(fold, bool) or not isinstance(fold, int) or fold < 0:\n        raise ValueError("Preparation must specify a nonnegative integer fold")\n    manifest_hash = sha256(manifest)\n    expected = preparation.get("manifest_sha256", preparation.get("experiment_manifest_sha256"))\n    if expected is not None and expected != manifest_hash:\n        raise ValueError("Preparation manifest SHA256 mismatch")\n    rows, fields = _read_csv(manifest)\n    if UID not in fields or not rows:\n        raise ValueError("Prepared manifest must contain study rows and " + UID)\n    uids = [_component(row[UID]) for row in rows]\n    if len(set(uids)) != len(uids):\n        raise ValueError("Duplicate study in prepared manifest")\n    metadata, series_hash = {}, None\n    if (comp / "train_series.csv").exists():\n        series_rows, series_fields = _read_csv(comp / "train_series.csv")\n        if not {UID, "SeriesInstanceUID"}.issubset(series_fields):\n            raise ValueError("train_series.csv is missing UID columns")\n        for row in series_rows:\n            key = (row[UID], row["SeriesInstanceUID"])\n            if key in metadata and metadata[key] != row:\n                raise ValueError("Conflicting train_series.csv rows")\n            metadata[key] = row\n        series_hash = sha256(comp / "train_series.csv")\n    image_config = {**cfg, "comp_dir": str(comp), "version": VERSION}\n    identity = dict(config=image_config, study_ids=uids, series_metadata_sha256=series_hash)\n    image_id = fingerprint(identity)\n    out = Path(config["out_dir"]).resolve() / cfg["recipe"]\n    if out.is_relative_to(comp) or manifest.is_relative_to(out):\n        raise ValueError("Output must be separate from competition/preparation inputs")\n    out.mkdir(parents=True, exist_ok=True)\n    lock = out / ".build.lock"\n    try:\n        handle = lock.open("x")\n    except FileExistsError as exc:\n        raise RuntimeError("Cache already has a writer; inspect .build.lock before retrying") from exc\n    try:\n        with handle:\n            return _build(out, comp, manifest, preparation_path, preparation, rows, fields, uids,\n                          metadata, cfg, config, image_config, identity, image_id, manifest_hash,\n                          series_hash, started)\n    finally:\n        lock.unlink()\n\n\ndef _build(out, comp, manifest, preparation_path, preparation, rows, fields, uids,\n           metadata, cfg, config, image_config, identity, image_id, manifest_hash,\n           series_hash, started):\n    identity_path = out / "image_identity.json"\n    if identity_path.exists():\n        if json.loads(identity_path.read_text(encoding="utf-8")).get("image_id") != image_id:\n            raise ValueError("Cache config/cohort mismatch; choose a new out_dir")\n    elif any(p.name != ".build.lock" for p in out.iterdir()):\n        raise ValueError("Nonempty cache has no verifiable identity")\n    count = min(len(rows), cfg["max_studies"] or len(rows))\n    per_study = 3 * cfg["slices"] * (cfg["size"]**2 + 1) + 65536\n    estimate = math.ceil(per_study * count * 1.05)\n    budget = float(config.get("max_cache_gb", 30))\n    if not math.isfinite(budget) or budget <= 0 or estimate > budget * 1e9:\n        raise OSError("Estimated cache exceeds max_cache_gb")\n    existing = sum(p.stat().st_size for p in (out / "studies").glob("*.npz"))\n    if shutil.disk_usage(out).free < max(per_study * 2, estimate - existing) + 64 * 1024**2:\n        raise OSError("Insufficient free disk for bounded estimate and working reserve")\n    write_json(identity_path, {**identity, "image_id": image_id})\n    meta = dict(schema=SCHEMA, cache_id=None, image_id=image_id, cohort_id=preparation["cohort_id"],\n                supervision_id=preparation["supervision_id"], fold=preparation.get("fold", preparation.get("config", {}).get("fold")),\n                config={**image_config, "manifest_path": str(manifest), "out_dir": str(out)},\n                complete=False, estimated_bytes=estimate, source_manifest_sha256=manifest_hash,\n                preparation_sha256=sha256(preparation_path), planes=list(PLANES),\n                geometry="native in-plane orientation; geometrically ordered sampled slice stack; not isotropic",\n                normalization_scope="selected series stack", environment=dict(python=platform.python_version(), pydicom=pydicom.__version__, numpy=np.__version__))\n    write_json(out / "cache_meta.json", meta)\n    _csv(out / "cache_manifest.csv", rows, fields)\n    cache = out / "studies"\n    cache.mkdir(exist_ok=True)\n    index, audits, source_hashes = [], [], {}\n    resumed = failed = 0\n    for number, uid in enumerate(uids):\n        if number >= count:\n            index.append({UID: uid, "file": "", "sha256": ""})\n            audits.append({UID: uid, "status": "unprocessed", "errors": []})\n            continue\n        tick = time.monotonic()\n        name = fingerprint(uid) + ".npz"\n        path, sidecar = cache / name, cache / (name + ".json")\n        try:\n            folders = _folders(comp, uid)\n            signature = _source_hash(folders, comp)\n            source_hashes[uid] = signature\n            info = _reuse(path, sidecar, image_id, signature, cfg)\n            if info is not None:\n                resumed += 1\n            else:\n                images, mask, info = _study(folders, metadata, uid, cfg)\n                temporary = path.with_suffix(".npz.tmp")\n                with temporary.open("wb") as stream:\n                    np.savez_compressed(stream, images=images, mask=mask)\n                temporary.replace(path)\n                info.update(image_id=image_id, source_hash=signature, sha256=sha256(path))\n                write_json(sidecar, info)\n            index.append({UID: uid, "file": path.relative_to(out).as_posix(), "sha256": info["sha256"]})\n            audits.append({UID: uid, "status": "ok" if info["success"] else "failed", **info})\n            failed += int(not info["success"])\n        except Exception as exc:\n            failed += 1\n            index.append({UID: uid, "file": "", "sha256": ""})\n            audits.append({UID: uid, "status": "failed", "errors": [f"{type(exc).__name__}: {exc}"]})\n        if audits[-1]["status"] == "failed":\n            LOG.error("Study %s failed: %s", uid, audits[-1]["errors"])\n        elif audits[-1].get("errors"):\n            LOG.warning("Study %s series audit: %s", uid, audits[-1]["errors"])\n        if number == 0 or (number + 1) % 25 == 0 or number + 1 == count:\n            LOG.info("MRI cache %s %d/%d; failed=%d resumed=%d study=%.2fs elapsed=%.1fs", cfg["recipe"], number + 1, count, failed, resumed, time.monotonic()-tick, time.monotonic()-started)\n    _csv(out / "cache_index.csv", index, [UID, "file", "sha256"])\n    write_json(out / "cache_audit.json", audits)\n    meta.update(manifest_sha256=sha256(out / "cache_manifest.csv"), index_sha256=sha256(out / "cache_index.csv"),\n                complete=failed == 0 and count == len(rows),\n                counts=dict(studies=len(rows), processed=count, failed=failed, unprocessed=len(rows)-count, resumed=resumed),\n                seconds=round(time.monotonic()-started, 3), source_hashes=source_hashes)\n    meta["cache_id"] = fingerprint(dict(config=image_config, source_hashes=source_hashes,\n                                       series_metadata_sha256=series_hash, manifest_sha256=manifest_hash,\n                                       preparation_sha256=meta["preparation_sha256"]))\n    write_json(out / "cache_meta.json", meta)\n    return out\n', 'registry.py': '"""One-factor experiments. Each run is selected explicitly; no automatic sweep."""\nBASE = dict(model=\'resnet18\', pooling=\'meanmax\', loss=\'bce\', pretrained=True,\n            strategy=\'finetune\', epochs=8, batch_size=2, accumulate=4,\n            lr=3e-4, encoder_lr=3e-5, weight_decay=1e-4, dropout=0.2,\n            seed=42, workers=0, train_slices=8, eval_slices=16,\n            augmentation=\'none\', amp=True, patience=3, min_epochs=4,\n            selection=\'macro_bce\', gamma=2.0, gamma_pos=0.0, gamma_neg=4.0,\n            asl_clip=0.05, ranking_weight=0.1, label_smoothing=0.05,\n            planes=\'all\', allow_downloads=False, weights_path=None)\n\nEXPERIMENTS = {\n    \'r00_resnet_control\': {},\n    \'r01_target_attention\': {\'pooling\': \'attention\'},\n    \'r02_single_sagittal\': {\'planes\': \'sagittal\'},\n    \'r03_dense_slices\': {\'train_slices\': 16},\n    \'r04_frozen_encoder\': {\'strategy\': \'frozen\'},\n    \'r05_staged_unfreeze\': {\'strategy\': \'staged\'},\n    \'r06_monai_intensity_aug\': {\'augmentation\': \'intensity\'},\n    \'l01_pos_weight_bce\': {\'loss\': \'pos_weight\'},\n    \'l02_monai_focal\': {\'loss\': \'focal\'},\n    \'l03_asymmetric\': {\'loss\': \'asl\'},\n    \'l04_bce_rank\': {\'loss\': \'bce_rank\'},\n    \'l05_label_smoothing\': {\'loss\': \'smooth\'},\n    \'h01_lower_lr\': {\'lr\': 1e-4, \'encoder_lr\': 1e-5},\n    \'h02_higher_decay\': {\'weight_decay\': 1e-3},\n    \'h03_longer_budget\': {\'epochs\': 16, \'min_epochs\': 8, \'patience\': 5},\n    \'h04_200epoch_early_stop\': {\'epochs\': 200, \'min_epochs\': 8, \'patience\': 12},\n    \'m01_dinov2_frozen\': {\'model\': \'dinov2\', \'strategy\': \'frozen\'},\n    \'m02_monai_3d\': {\'model\': \'monai3d\', \'pretrained\': False, \'train_slices\': 16},\n    \'s00_tiny_smoke\': {\'model\': \'tiny\', \'pretrained\': False, \'epochs\': 2,\n                      \'min_epochs\': 1, \'batch_size\': 2, \'accumulate\': 1},\n}\n\n\ndef experiment(name, **overrides):\n    if name not in EXPERIMENTS:\n        raise ValueError(f\'Unknown experiment {name}; choose from {list(EXPERIMENTS)}\')\n    unknown = set(overrides) - set(BASE) - {\'cache_dir\', \'out_dir\', \'resume_path\'}\n    if unknown:\n        raise ValueError(f\'Unknown settings: {sorted(unknown)}\')\n    return {**BASE, **EXPERIMENTS[name], **overrides, \'experiment\': name}\n', 'models.py': '"""Shared 2D encoder MIL, frozen SSL encoder, and small MONAI 3D comparator."""\nfrom pathlib import Path\nimport torch\nfrom torch import nn\nfrom .common import sha256\n\n\nclass StudyModel(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = cfg\n        self.weight_hash = None\n        family = cfg[\'model\']\n        if family == \'tiny\':\n            self.encoder = nn.Sequential(nn.Conv2d(3, 16, 3, 2, 1), nn.ReLU(),\n                                         nn.Conv2d(16, 32, 3, 2, 1), nn.ReLU(),\n                                         nn.AdaptiveAvgPool2d(1), nn.Flatten(1))\n            dim = 32\n        elif family == \'resnet18\':\n            from torchvision.models import resnet18, ResNet18_Weights\n            self.encoder = resnet18(weights=None)\n            if cfg[\'pretrained\']:\n                if cfg.get(\'weights_path\'):\n                    self.encoder.load_state_dict(torch.load(cfg[\'weights_path\'], map_location=\'cpu\', weights_only=True))\n                    self.weight_hash = sha256(cfg[\'weights_path\'])\n                elif cfg.get(\'allow_downloads\'):\n                    weights = ResNet18_Weights.IMAGENET1K_V1\n                    self.encoder.load_state_dict(weights.get_state_dict(progress=True, check_hash=True))\n                    self.weight_hash = sha256(Path(torch.hub.get_dir())/\'checkpoints\'/weights.url.rsplit(\'/\', 1)[-1])\n                else:\n                    raise FileNotFoundError(\'Attach resnet18_imagenet.pth and set weights_path, or enable Internet and allow_downloads.\')\n            dim = self.encoder.fc.in_features\n            self.encoder.fc = nn.Identity()\n        elif family == \'dinov2\':\n            import timm\n            self.encoder = timm.create_model(\'vit_small_patch14_dinov2.lvd142m\', pretrained=False, img_size=224)\n            if not cfg.get(\'weights_path\'):\n                raise FileNotFoundError(\'DINO requires dinov2_vits14_timm.pth from the notebook weight-preparation cell.\')\n            self.encoder.load_state_dict(torch.load(cfg[\'weights_path\'], map_location=\'cpu\', weights_only=True))\n            self.weight_hash = sha256(cfg[\'weights_path\'])\n            dim = self.encoder.num_features\n        elif family == \'monai3d\':\n            from monai.networks.nets import DenseNet\n            self.encoder = DenseNet(spatial_dims=3, in_channels=1, out_channels=128,\n                                    init_features=16, growth_rate=8, block_config=(2, 2, 2), bn_size=2)\n            dim = 128\n        else:\n            raise ValueError(f\'Unknown model: {family}\')\n        # Plane identity is retained rather than treating sagittal/coronal/axial interchangeably.\n        self.plane_embedding = nn.Embedding(3, dim)\n        self.attention = nn.Sequential(nn.Linear(dim, 128), nn.Tanh(), nn.Linear(128, 12))\n        self.dropout = nn.Dropout(cfg[\'dropout\'])\n        self.head = nn.Linear(dim * 2 if cfg[\'pooling\'] == \'meanmax\' else dim, 12)\n        self.encoder_frozen = False\n\n    def set_epoch(self, epoch):\n        frozen = self.cfg[\'strategy\'] == \'frozen\' or (self.cfg[\'strategy\'] == \'staged\' and epoch < 2)\n        self.encoder_frozen = frozen\n        self.encoder.requires_grad_(not frozen)\n        # Keep BatchNorm fixed with tiny study batches. Other encoder modules can train.\n        if frozen:\n            self.encoder.eval()\n        for module in self.encoder.modules():\n            if isinstance(module, nn.modules.batchnorm._BatchNorm):\n                module.eval()\n\n    def forward(self, images, mask):\n        # images B,planes,slices,H,W; mask B,planes,slices\n        b, planes, depth, h, w = images.shape\n        if not mask.flatten(1).any(1).all():\n            raise ValueError(\'Study has no valid MRI slice\')\n        if self.cfg[\'model\'] == \'monai3d\':\n            valid = mask.any(-1).reshape(-1)\n            x = images.reshape(b*planes, 1, depth, h, w)\n            x = x * mask.reshape(b*planes, 1, depth, 1, 1)\n            features = self.encoder(x[valid])\n            all_features = features.new_zeros((b*planes, features.shape[-1]))\n            all_features[valid] = features\n            features = all_features.reshape(b, planes, -1)\n            token_mask = mask.any(-1)\n            plane_ids = torch.arange(planes, device=images.device)\n        else:\n            valid = mask.reshape(-1)\n            x = images.reshape(b*planes*depth, 1, h, w)[valid].repeat(1, 3, 1, 1)\n            if self.cfg[\'model\'] == \'dinov2\':\n                x = nn.functional.interpolate(x, size=(224, 224), mode=\'bilinear\', align_corners=False)\n            if self.cfg[\'pretrained\']:\n                mean = x.new_tensor([.485, .456, .406])[None, :, None, None]\n                std = x.new_tensor([.229, .224, .225])[None, :, None, None]\n                x = (x-mean)/std\n            # Frozen encoders use no autograd; no claim of precomputed feature caching.\n            with torch.set_grad_enabled(self.training and not self.encoder_frozen):\n                encoded = self.encoder(x)\n            all_features = encoded.new_zeros((b*planes*depth, encoded.shape[-1]))\n            all_features[valid] = encoded\n            features = all_features.reshape(b, planes*depth, -1)\n            token_mask = mask.reshape(b, planes*depth)\n            plane_ids = torch.arange(planes, device=images.device).repeat_interleave(depth)\n        features = features + self.plane_embedding(plane_ids)[None]\n        if self.cfg[\'pooling\'] == \'meanmax\':\n            mean = (features*token_mask[..., None]).sum(1)/token_mask.sum(1, keepdim=True)\n            maximum = features.masked_fill(~token_mask[..., None], -torch.inf).amax(1)\n            return self.head(self.dropout(torch.cat([mean, maximum], -1)))\n        scores = self.attention(features).masked_fill(~token_mask[..., None], -torch.inf)\n        pooled = torch.einsum(\'bnt,bnf->btf\', scores.softmax(1), features)\n        return (self.dropout(pooled)*self.head.weight[None]).sum(-1)+self.head.bias\n', 'losses.py': '"""Masked per-target objectives; validation always uses the same plain BCE."""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\nclass MaskedObjective(nn.Module):\n    def __init__(self, cfg, train_y, train_w):\n        super().__init__()\n        self.cfg = cfg\n        active = train_w.sum(0) > 0\n        if not active.any():\n            raise ValueError(\'No active labels in training\')\n        self.register_buffer(\'active\', active)\n        # Fixed training-set denominators avoid changing target scale with sparse minibatches.\n        self.register_buffer(\'mean_weight\', train_w.mean(0).clamp_min(1e-8))\n        pos = (train_y*train_w).sum(0)\n        neg = ((1-train_y)*train_w).sum(0)\n        self.register_buffer(\'pos_weight\', ((neg+1)/(pos+1)).clamp(.25, 4))\n        if cfg[\'loss\'] == \'focal\':\n            from monai.losses import FocalLoss\n            self.focal = FocalLoss(gamma=cfg[\'gamma\'], reduction=\'none\', use_softmax=False)\n\n    def forward(self, logits, y, w):\n        logits = logits.float()\n        kind = self.cfg[\'loss\']\n        if kind == \'focal\':\n            raw = self.focal(logits, y)\n        elif kind == \'asl\':\n            p = logits.sigmoid()\n            pn = (1-p+self.cfg[\'asl_clip\']).clamp(max=1)\n            raw = -(y*(1-p).pow(self.cfg[\'gamma_pos\'])*p.clamp_min(1e-8).log()\n                    + (1-y)*(1-pn).pow(self.cfg[\'gamma_neg\'])*pn.clamp_min(1e-8).log())\n        else:\n            target = y*(1-self.cfg[\'label_smoothing\'])+.5*self.cfg[\'label_smoothing\'] if kind == \'smooth\' else y\n            raw = F.binary_cross_entropy_with_logits(logits, target, reduction=\'none\',\n                        pos_weight=self.pos_weight if kind == \'pos_weight\' else None)\n        loss = ((raw*w).mean(0)/self.mean_weight)[self.active].mean()\n        if kind == \'bce_rank\':\n            pairs = []\n            for j in range(y.shape[1]):\n                positive = logits[(y[:, j] == 1) & (w[:, j] > 0), j]\n                negative = logits[(y[:, j] == 0) & (w[:, j] > 0), j]\n                if positive.numel() and negative.numel():\n                    pairs.append(F.softplus(-(positive[:, None]-negative[None, :])).mean())\n            if pairs:\n                loss = loss + self.cfg[\'ranking_weight\']*torch.stack(pairs).mean()\n        return loss\n', 'training.py': '"""Study-level training with provenance, fixed validation, priors and resume."""\nfrom pathlib import Path\nimport json\nimport math\nimport random\nimport time\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset, DataLoader\nfrom sklearn.metrics import (roc_auc_score, average_precision_score,\n                              brier_score_loss, balanced_accuracy_score, f1_score)\nfrom .common import UID, TARGETS, YC, WC, fingerprint, sha256, write_json, safe_output, environment\nfrom .models import StudyModel\nfrom .losses import MaskedObjective\n\n\nclass CacheDataset(Dataset):\n    def __init__(self, root, frame, index, cfg, training=False):\n        self.root, self.frame, self.cfg, self.training = Path(root), frame, cfg, training\n        self.files = index.set_index(UID)[\'file\'].to_dict()\n        self.gamma = None\n        if training and cfg[\'augmentation\'] == \'intensity\':\n            from monai.transforms import RandAdjustContrast\n            self.gamma = RandAdjustContrast(prob=.5, gamma=(.85, 1.15))\n            self.gamma.set_random_state(seed=cfg[\'seed\'])\n\n    def __len__(self):\n        return len(self.frame)\n\n    def __getitem__(self, i):\n        row = self.frame.iloc[i]\n        with np.load(self.root/self.files[row[UID]], allow_pickle=False) as data:\n            images, mask = data[\'images\'].astype(np.float32)/255, data[\'mask\'].astype(bool)\n        n = self.cfg[\'train_slices\' if self.training else \'eval_slices\']\n        selected = np.zeros((3, n, *images.shape[-2:]), np.float32)\n        selected_mask = np.zeros((3, n), bool)\n        for plane in range(3):\n            if self.cfg[\'planes\'] == \'sagittal\' and plane != 0:\n                continue\n            valid = np.flatnonzero(mask[plane])\n            if len(valid) == 0:\n                continue\n            # Uniform bins with random within-bin sampling during training; fixed eval anchors.\n            count = min(n, len(valid))\n            if self.training:\n                bounds = np.linspace(0, len(valid), count+1).astype(int)\n                ids = np.array([np.random.randint(bounds[j], bounds[j+1]) for j in range(count)])\n            else:\n                ids = np.linspace(0, len(valid)-1, count).round().astype(int)\n            selected[plane, :count] = images[plane, valid[ids]]\n            selected_mask[plane, :count] = True\n        x = torch.from_numpy(selected)\n        if self.gamma is not None:\n            # Shared intensity transform preserves slice order; no flips without laterality audit.\n            x = torch.as_tensor(self.gamma(x)).float().clamp(0, 1)\n            x *= torch.from_numpy(selected_mask)[..., None, None]\n        return x, torch.from_numpy(selected_mask), torch.tensor(row[YC].to_numpy(float), dtype=torch.float32), torch.tensor(row[WC].to_numpy(float), dtype=torch.float32)\n\n\ndef metrics(y, w, p, active, prior=None):\n    p = np.asarray(p, dtype=np.float64)\n    if not np.isfinite(p).all() or (p < 0).any() or (p > 1).any():\n        raise ValueError(\'Nonfinite or out-of-range predictions\')\n    p = np.clip(p, 1e-7, 1-1e-7)\n    per_target, bces, aucs, aps, briers, balanced_accs, f1s = [], [], [], [], [], [], []\n    for j, name in enumerate(TARGETS):\n        observed = w[:, j] > 0\n        binary = observed & np.isin(y[:, j], [0, 1])\n        eligible = bool(active[j] and len(np.unique(y[binary, j])) == 2)\n        bce = float(((-y[:, j]*np.log(p[:, j])-(1-y[:, j])*np.log1p(-p[:, j]))*w[:, j]).sum()/w[:, j].sum()) if active[j] and observed.any() else None\n        auc = float(roc_auc_score(y[binary, j], p[binary, j])) if eligible else None\n        ap = float(average_precision_score(y[binary, j], p[binary, j])) if eligible else None\n        brier = float(brier_score_loss(y[binary, j], p[binary, j])) if eligible else None\n        balanced_accuracy = (float(balanced_accuracy_score(y[binary, j], p[binary, j] >= 0.5)) if eligible else None)\n        f1 = float(f1_score(y[binary, j], p[binary, j] >= 0.5, zero_division=0)) if eligible else None\n        if bce is not None:\n            bces.append(bce)\n        if auc is not None:\n            aucs.append(auc)\n        if ap is not None: aps.append(ap)\n        if brier is not None: briers.append(brier)\n        if balanced_accuracy is not None: balanced_accs.append(balanced_accuracy)\n        if f1 is not None: f1s.append(f1)\n        per_target.append(dict(target=name, n_positive=int(((y[:, j] == 1)&observed).sum()),\n                               n_negative=int(((y[:, j] == 0)&observed).sum()),\n                               n_supervised=int(observed.sum()), train_active=bool(active[j]), bce=bce, auc=auc, ap=ap))\n        per_target[-1].update(brier=brier, balanced_accuracy=balanced_accuracy, f1_at_05=f1)\n    result = {\'macro_bce\': float(np.mean(bces)) if bces else None,\n              \'macro_auc\': float(np.mean(aucs)) if aucs else None,\n              \'macro_ap\': float(np.mean(aps)) if aps else None,\n              \'macro_brier\': float(np.mean(briers)) if briers else None,\n              \'macro_balanced_accuracy\': float(np.mean(balanced_accs)) if balanced_accs else None,\n              \'macro_f1_at_05\': float(np.mean(f1s)) if f1s else None,\n              \'n_metric_targets\': len(aucs),\n              \'eligible_auc_targets\': [r[\'target\'] for r in per_target if r[\'auc\'] is not None],\n              \'n_auc_targets\': len(aucs), \'per_target\': per_target}\n    if prior is not None:\n        baseline = metrics(y, w, np.broadcast_to(prior, p.shape), active)\n        result[\'constant_macro_bce\'] = baseline[\'macro_bce\']\n        result[\'delta_vs_constant\'] = result[\'macro_bce\']-baseline[\'macro_bce\'] if bces else None\n    return result\n\n\ndef _save_checkpoint(path, obj):\n    path = Path(path)\n    temp = path.with_suffix(\'.tmp\')\n    torch.save(obj, temp)\n    temp.replace(path)\n\n\ndef run_training(cfg):\n    cfg = dict(cfg)\n    for field in [\'epochs\', \'batch_size\', \'accumulate\', \'train_slices\', \'eval_slices\']:\n        if not isinstance(cfg[field], int) or isinstance(cfg[field], bool) or cfg[field] < 1:\n            raise ValueError(f\'{field} must be a positive integer\')\n    if cfg[\'workers\'] != 0:\n        raise ValueError(\'Use workers=0 for reproducible notebook resume and MONAI transform RNG.\')\n    if cfg[\'selection\'] not in [\'macro_bce\', \'macro_auc\']:\n        raise ValueError(\'selection must be macro_bce or macro_auc\')\n    cache = Path(cfg[\'cache_dir\'])\n    meta = json.loads((cache/\'cache_meta.json\').read_text())\n    if not meta[\'complete\'] or meta.get(\'config\', {}).get(\'max_studies\', 0) > 0:\n        counts = meta.get(\'counts\', {})\n        if meta.get(\'config\', {}).get(\'max_studies\', 0) > 0 or counts.get(\'unprocessed\', 0):\n            raise ValueError(\n                \'This is a preprocessing PILOT/truncated cache and cannot be trained. \'\n                \'Rerun notebook 02 with PILOT=False, Save & Run All, and attach that full output. \'\n                f"Processed={counts.get(\'processed\', \'?\')}, unprocessed={counts.get(\'unprocessed\', \'?\')}."\n            )\n        raise ValueError(\n            \'Incomplete cache: inspect cache_audit.json and decode_errors.csv, fix the failures, \'\n            \'then rebuild the full cache before training.\'\n        )\n    frame = pd.read_csv(cache/\'cache_manifest.csv\', dtype={UID: str, \'group\': str})\n    index = pd.read_csv(cache/\'cache_index.csv\', dtype={UID: str})\n    if sha256(cache/\'cache_manifest.csv\') != meta[\'manifest_sha256\'] or sha256(cache/\'cache_index.csv\') != meta[\'index_sha256\']:\n        raise ValueError(\'Cache manifest/index hash mismatch\')\n    if frame[UID].duplicated().any() or index[UID].duplicated().any() or set(frame[UID]) != set(index[UID]):\n        raise ValueError(\'Cache studies do not match the fixed cohort exactly\')\n    if frame[[UID, \'group\', \'fold\']].isna().any().any() or (frame.groupby(\'group\').fold.nunique() != 1).any():\n        raise ValueError(\'Missing identifiers or split group\')\n    values = frame[YC+WC].to_numpy(float)\n    if not np.isfinite(values).all() or (values < 0).any() or (values > 1).any():\n        raise ValueError(\'Labels and weights must be finite and within [0,1]\')\n    for _, row in index.iterrows():\n        path = (cache/row[\'file\']).resolve()\n        if not path.is_relative_to(cache.resolve()) or not path.is_file() or sha256(path) != row[\'sha256\']:\n            raise ValueError(f\'Missing or changed cache file for {row[UID]}\')\n        with np.load(path, allow_pickle=False) as data:\n            images, mask = data[\'images\'], data[\'mask\']\n            if images.ndim != 4 or images.shape[0] != 3 or mask.shape != images.shape[:2] or not mask.any():\n                raise ValueError(f\'Invalid cache shape/mask for {row[UID]}\')\n            if cfg[\'planes\'] == \'sagittal\' and not mask[0].any():\n                raise ValueError(\'Sagittal-only comparison needs sagittal data for every fixed cohort study; cohort will not be silently filtered.\')\n    tr = frame[frame.fold != meta[\'fold\']].sort_values(UID).reset_index(drop=True)\n    va = frame[frame.fold == meta[\'fold\']].sort_values(UID).reset_index(drop=True)\n    if tr.empty or va.empty or set(tr.group)&set(va.group):\n        raise ValueError(\'Empty train/validation or overlapping groups\')\n    if cfg[\'model\'] == \'monai3d\' and min(cfg[\'train_slices\'], cfg[\'eval_slices\']) < 16:\n        raise ValueError(\'MONAI 3D needs at least 16 retained slice positions; set train_slices=16.\')\n    random.seed(cfg[\'seed\']); np.random.seed(cfg[\'seed\']); torch.manual_seed(cfg[\'seed\'])\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(cfg[\'seed\'])\n    torch.backends.cudnn.benchmark = False\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    model = StudyModel(cfg).to(device)\n    train_y, train_w = torch.tensor(tr[YC].to_numpy(float), dtype=torch.float32), torch.tensor(tr[WC].to_numpy(float), dtype=torch.float32)\n    objective = MaskedObjective(cfg, train_y, train_w).to(device)\n    active = objective.active.cpu().numpy()\n    prior = (train_y*train_w).sum(0)/train_w.sum(0).clamp_min(1e-8)\n    prior[~torch.tensor(active)] = .5\n    prior = prior.numpy().clip(1e-7, 1-1e-7)\n    y, w = va[YC].to_numpy(float), va[WC].to_numpy(float)\n    constant = metrics(y, w, np.broadcast_to(prior, y.shape), active)\n    if constant[\'macro_bce\'] is None:\n        raise ValueError(\'No validation supervision for active training targets\')\n    if cfg[\'selection\'] == \'macro_auc\' and constant[\'n_auc_targets\'] != 12:\n        raise ValueError(\'AUC selection requires positive and negative validation examples for all 12 targets. Use BCE for exploratory sparse labels.\')\n    source_id = fingerprint({p.name: sha256(p) for p in Path(__file__).parent.glob(\'*.py\')})\n    identity_config = {k: v for k, v in cfg.items() if k not in [\'out_dir\', \'resume_path\', \'cache_dir\', \'weights_path\', \'allow_downloads\']}\n    run_id = fingerprint({\'config\': identity_config, \'source_id\': source_id,\n                          \'cache_id\': meta[\'cache_id\'], \'weights\': model.weight_hash})\n    out = safe_output(cfg[\'out_dir\'], run_id)\n    if (out/\'run_summary.json\').exists():\n        print(\'Completed identical run already exists:\', out)\n        return out\n    write_json(out/\'config.json\', cfg)\n    validation_id = fingerprint(va[[UID, \'group\', \'fold\']+YC+WC].to_dict(\'records\'))\n    optimizer = torch.optim.AdamW([\n        {\'params\': model.encoder.parameters(), \'lr\': cfg[\'encoder_lr\']},\n        {\'params\': [p for name, p in model.named_parameters() if not name.startswith(\'encoder.\')], \'lr\': cfg[\'lr\']}], weight_decay=cfg[\'weight_decay\'])\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg[\'epochs\'])\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=cfg[\'amp\'] and device.type == \'cuda\')\n    train_ds = CacheDataset(cache, tr, index, cfg, True)\n    val_ds = CacheDataset(cache, va, index, cfg, False)\n    loader_rng = torch.Generator().manual_seed(cfg[\'seed\'])\n    train_loader = DataLoader(train_ds, batch_size=cfg[\'batch_size\'], shuffle=True, generator=loader_rng, num_workers=0, pin_memory=device.type == \'cuda\')\n    val_loader = DataLoader(val_ds, batch_size=cfg[\'batch_size\'], num_workers=0, pin_memory=device.type == \'cuda\')\n    start_epoch, stale, best, best_epoch, history = 0, 0, float(\'inf\'), 0, []\n    elapsed_before = 0.0\n    resume = cfg.get(\'resume_path\') or (str(out/\'last.pt\') if (out/\'last.pt\').exists() else None)\n    best_predictions = None\n    if resume:\n        # Full optimizer/RNG checkpoint is trusted only from this user\'s experiment output.\n        ck = torch.load(resume, map_location=\'cpu\', weights_only=False)\n        if ck[\'run_id\'] != run_id:\n            raise ValueError(\'Resume config/cache/source/weights mismatch. Use matching inputs or a new experiment.\')\n        model.load_state_dict(ck[\'model\']); optimizer.load_state_dict(ck[\'optimizer\'])\n        scheduler.load_state_dict(ck[\'scheduler\']); scaler.load_state_dict(ck[\'scaler\'])\n        start_epoch, stale, best, best_epoch, history = ck[\'epoch\'], ck[\'stale\'], ck[\'best\'], ck[\'best_epoch\'], ck[\'history\']\n        elapsed_before = ck[\'elapsed\']\n        torch.set_rng_state(ck[\'torch_rng\']); np.random.set_state(ck[\'numpy_rng\']); random.setstate(ck[\'python_rng\'])\n        loader_rng.set_state(ck[\'loader_rng\'])\n        if device.type == \'cuda\' and ck[\'cuda_rng\'] is not None:\n            torch.cuda.set_rng_state_all(ck[\'cuda_rng\'])\n        if train_ds.gamma is not None:\n            train_ds.gamma.R.set_state(ck[\'augmentation_rng\'])\n        best_predictions = ck[\'best_predictions\']\n        # A resume must carry the selected best model as well as last state.\n        _save_checkpoint(out/\'best.pt\', ck[\'best_checkpoint\'])\n    else:\n        ck = {}\n    best_checkpoint = ck.get(\'best_checkpoint\')\n    print(\'Device:\', device, \'train:\', len(tr), \'validation:\', len(va), \'constant macro BCE:\', constant[\'macro_bce\'], flush=True)\n    print(\'Eligible AUC targets:\', constant[\'n_auc_targets\'], \'/12. Labels remain report-derived.\', flush=True)\n    wall_start = time.perf_counter()\n    for epoch in range(start_epoch, cfg[\'epochs\']):\n        if stale >= cfg[\'patience\'] and epoch >= cfg[\'min_epochs\']:\n            break\n        model.train(); model.set_epoch(epoch)\n        optimizer.zero_grad(set_to_none=True)\n        train_total = 0.0\n        group_denominator = 1\n        for step, (x, mask, target, weight) in enumerate(train_loader):\n            if step % cfg[\'accumulate\'] == 0:\n                group_start = step*cfg[\'batch_size\']\n                group_denominator = min(cfg[\'accumulate\']*cfg[\'batch_size\'], len(tr)-group_start)\n            x, mask, target, weight = x.to(device), mask.to(device), target.to(device), weight.to(device)\n            with torch.autocast(device_type=device.type, enabled=cfg[\'amp\'] and device.type == \'cuda\'):\n                logits = model(x, mask)\n                raw_loss = objective(logits, target, weight)\n                loss = raw_loss * len(x)/group_denominator\n            if not torch.isfinite(loss):\n                raise ValueError(\'Nonfinite training loss\')\n            scaler.scale(loss).backward()\n            train_total += raw_loss.item()*len(x)\n            if (step+1) % cfg[\'accumulate\'] == 0 or step+1 == len(train_loader):\n                scaler.unscale_(optimizer)\n                nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)\n        scheduler.step()\n        model.eval(); predictions = []\n        with torch.no_grad():\n            for x, mask, _, _ in val_loader:\n                with torch.autocast(device_type=device.type, enabled=cfg[\'amp\'] and device.type == \'cuda\'):\n                    pred = model(x.to(device), mask.to(device))\n                predictions.append(pred.float().sigmoid().cpu().numpy())\n        predictions = np.concatenate(predictions)\n        measured = metrics(y, w, predictions, active, prior)\n        criterion = measured[\'macro_bce\'] if cfg[\'selection\'] == \'macro_bce\' else -measured[\'macro_auc\']\n        elapsed = elapsed_before + time.perf_counter()-wall_start\n        record = dict(epoch=epoch+1, train_objective=train_total/len(tr), val_macro_bce=measured[\'macro_bce\'],\n                      val_macro_auc=measured[\'macro_auc\'], constant_macro_bce=constant[\'macro_bce\'],\n                      delta_vs_constant=measured[\'delta_vs_constant\'], seconds=elapsed)\n        history.append(record); print(record, flush=True)\n        if criterion < best:\n            best, best_epoch, stale = criterion, epoch+1, 0\n            best_predictions = predictions.copy()\n            best_checkpoint = {\'model\': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},\n                               \'config\': cfg, \'run_id\': run_id, \'cache_metadata\': meta, \'targets\': TARGETS, \'epoch\': best_epoch}\n            _save_checkpoint(out/\'best.pt\', best_checkpoint)\n        else:\n            stale += 1\n        _save_checkpoint(out/\'last.pt\', dict(run_id=run_id, model=model.state_dict(), optimizer=optimizer.state_dict(),\n            scheduler=scheduler.state_dict(), scaler=scaler.state_dict(), epoch=epoch+1,\n            best=best, best_epoch=best_epoch, stale=stale, history=history, elapsed=elapsed,\n            best_predictions=best_predictions, best_checkpoint=best_checkpoint,\n            torch_rng=torch.get_rng_state(), numpy_rng=np.random.get_state(), python_rng=random.getstate(),\n            cuda_rng=torch.cuda.get_rng_state_all() if device.type == \'cuda\' else None, loader_rng=loader_rng.get_state(),\n            augmentation_rng=train_ds.gamma.R.get_state() if train_ds.gamma is not None else None))\n        pd.DataFrame(history).to_csv(out/\'history.csv\', index=False)\n    if best_predictions is None:\n        raise ValueError(\'No completed epoch/checkpoint\')\n    result = va[[UID, \'group\', \'fold\']+YC+WC].copy()\n    for j in range(12):\n        result[f\'p_{j}\'] = best_predictions[:, j]\n    result.to_csv(out/\'validation_predictions.csv\', index=False)\n    measured = metrics(y, w, best_predictions, active, prior)\n    pd.DataFrame(measured[\'per_target\']).to_csv(out/\'target_metrics.csv\', index=False)\n    summary = dict(run_id=run_id, experiment=cfg[\'experiment\'], validation_id=validation_id,\n        cohort_id=meta[\'cohort_id\'], supervision_id=meta[\'supervision_id\'], cache_id=meta[\'cache_id\'],\n        best_epoch=best_epoch, metrics=measured, active_targets=active.tolist(),\n        wall_seconds=elapsed_before+time.perf_counter()-wall_start, role=\'report_label_exploration\',\n        source_id=source_id, pretrained_sha256=model.weight_hash, environment=environment(),\n        validation_predictions_sha256=sha256(out/\'validation_predictions.csv\'),\n        best_checkpoint_sha256=sha256(out/\'best.pt\'), train_prior=prior.tolist())\n    write_json(out/\'run_summary.json\', summary)\n    return out\n', 'comparison.py': '"""Strict, exploratory comparisons of saved report-label validation predictions.\n\nNo training imports, ensemble fitting, class filling, or leaderboard inference.\n"""\nimport csv\nimport hashlib\nimport io\nimport json\nfrom itertools import combinations\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score\n\nfrom .common import TARGETS, UID, WC, YC, fingerprint, safe_output, sha256, write_json\n\nPC = [f\'p_{j}\' for j in range(12)]\nROLE = \'report_label_exploration\'\nMETRICS = [\'macro_bce\', \'macro_auc\', \'eligible_auc_targets\',\n           \'constant_macro_bce\', \'delta_vs_constant\']\nSUMMARY_KEYS = [\'run_id\', \'experiment\', \'validation_id\', \'cohort_id\',\n                \'supervision_id\', \'cache_id\', \'best_epoch\', \'wall_seconds\', \'role\']\nPAIR_COLUMNS = [\'reference_run_id\', \'run_id\', \'delta_macro_auc\', \'ci_low\', \'ci_high\',\n                \'bootstrap_requested\', \'bootstrap_valid\', \'bootstrap_discarded\']\nCORRELATION_COLUMNS = [\'run_a\', \'run_b\', \'target_index\', \'target\', \'n_rows\',\n                       \'pearson_r\', \'reason\']\n\n\ndef _load_run(directory):\n    directory = Path(directory)\n    # Hash precisely the bytes parsed, including when a producer is still running.\n    summary_bytes = (directory / \'run_summary.json\').read_bytes()\n    prediction_bytes = (directory / \'validation_predictions.csv\').read_bytes()\n    summary = json.loads(summary_bytes.decode(\'utf-8-sig\'))\n    if not isinstance(summary, dict):\n        raise ValueError(f\'{directory}: run_summary must be an object\')\n    missing = set(SUMMARY_KEYS + [\'active_targets\', \'metrics\']) - summary.keys()\n    if missing:\n        raise ValueError(f\'{directory}: missing summary keys {sorted(missing)}\')\n    if not isinstance(summary[\'metrics\'], dict) or set(METRICS) - summary[\'metrics\'].keys():\n        raise ValueError(f\'{directory}: incomplete metrics\')\n    for key in [\'run_id\', \'validation_id\', \'cohort_id\']:\n        if not isinstance(summary[key], str) or not summary[key].strip():\n            raise ValueError(f\'{directory}: invalid {key}\')\n    active = summary[\'active_targets\']\n    if not isinstance(active, list) or len(active) != 12 or any(type(v) is not bool for v in active):\n        raise ValueError(f\'{directory}: active_targets must contain 12 booleans\')\n    if summary[\'role\'] != ROLE:\n        raise ValueError(f\'{directory}: role must be {ROLE}\')\n    prediction_hash = hashlib.sha256(prediction_bytes).hexdigest()\n    if (\'validation_predictions_sha256\' in summary\n            and summary[\'validation_predictions_sha256\'] != prediction_hash):\n        raise ValueError(f\'{directory}: validation prediction checksum mismatch\')\n\n    decoded = prediction_bytes.decode(\'utf-8-sig\')\n    header = next(csv.reader(io.StringIO(decoded)), [])\n    required = [UID, \'group\', \'fold\'] + YC + WC + PC\n    if len(header) != len(set(header)) or set(required) - set(header):\n        raise ValueError(f\'{directory}: missing or duplicate prediction columns\')\n    frame = pd.read_csv(io.StringIO(decoded), dtype={UID: str, \'group\': str},\n                        keep_default_na=False, float_precision=\'round_trip\')\n    if frame.empty or frame[UID].duplicated().any():\n        raise ValueError(f\'{directory}: validation UIDs must be nonempty and unique\')\n    if any(frame[key].astype(str).str.strip().eq(\'\').any() for key in [UID, \'group\', \'fold\']):\n        raise ValueError(f\'{directory}: missing UID, group, or fold\')\n    for key in [\'fold\'] + YC + WC + PC:\n        frame[key] = pd.to_numeric(frame[key], errors=\'raise\')\n    values = frame[[\'fold\'] + YC + WC + PC].to_numpy(dtype=float)\n    if not np.isfinite(values).all():\n        raise ValueError(f\'{directory}: nonfinite prediction data\')\n    if (frame[\'fold\'] < 0).any() or (frame[\'fold\'] % 1 != 0).any():\n        raise ValueError(f\'{directory}: fold must be a nonnegative integer\')\n    if (frame.groupby(\'group\')[\'fold\'].nunique() > 1).any():\n        raise ValueError(f\'{directory}: a group spans multiple folds\')\n    if (frame[WC].to_numpy() < 0).any():\n        raise ValueError(f\'{directory}: negative weights\')\n    for columns, name in [(YC, \'labels\'), (PC, \'probabilities\')]:\n        a = frame[columns].to_numpy()\n        if ((a < 0) | (a > 1)).any():\n            raise ValueError(f\'{directory}: {name} must be in [0, 1]\')\n    return summary, frame.set_index(UID).sort_index(), {\n        \'run_id\': summary[\'run_id\'],\n        \'run_summary_sha256\': hashlib.sha256(summary_bytes).hexdigest(),\n        \'validation_predictions_sha256\': prediction_hash,\n    }\n\n\ndef _macro_auc(y, w, p, eligible):\n    """Return None if ANY fixed eligible target degenerates in this sample."""\n    aucs = []\n    for j in eligible:\n        mask = (w[:, j] > 0) & np.isin(y[:, j], [0, 1])\n        if np.unique(y[mask, j]).size != 2:\n            return None\n        # Weights gate observation, as in training\'s AUC; they do not weight ROC.\n        aucs.append(float(roc_auc_score(y[mask, j], p[mask, j])))\n    return float(np.mean(aucs)) if aucs else None\n\n\ndef _metrics(y, w, p, active, eligible):\n    p_bce = np.clip(p, 1e-7, 1 - 1e-7)\n    bces = []\n    for j in np.flatnonzero(active):\n        mask = w[:, j] > 0\n        if mask.any():\n            loss = -y[mask, j] * np.log(p_bce[mask, j]) - (1 - y[mask, j]) * np.log1p(-p_bce[mask, j])\n            bces.append(float(np.average(loss, weights=w[mask, j])))\n    return {\'macro_bce\': float(np.mean(bces)) if bces else None,\n            \'macro_auc\': _macro_auc(y, w, p, eligible),\n            \'eligible_auc_targets\': [TARGETS[j] for j in eligible],\n            \'n_auc_targets\': len(eligible)}\n\n\ndef _bootstrap_pairs(y, w, predictions, groups, eligible, summaries, measured, bootstrap, seed):\n    samples = [[] for _ in predictions[1:]]\n    unique_groups = np.unique(groups)\n    members = [np.flatnonzero(groups == group) for group in unique_groups]\n    rng = np.random.default_rng(seed)\n    if eligible:\n        for _ in range(bootstrap):\n            # Keep every row in each drawn group, including repeated group draws.\n            rows = np.concatenate([members[k] for k in rng.integers(len(members), size=len(members))])\n            scores = [_macro_auc(y[rows], w[rows], p[rows], eligible) for p in predictions]\n            if any(score is None for score in scores):\n                continue\n            for deltas, score in zip(samples, scores[1:]):\n                deltas.append(score - scores[0])\n    pairs = []\n    for i, deltas in enumerate(samples, start=1):\n        interval = np.quantile(deltas, [.025, .975]).tolist() if deltas else [None, None]\n        pairs.append(dict(reference_run_id=summaries[0][\'run_id\'], run_id=summaries[i][\'run_id\'],\n                          delta_macro_auc=(measured[i][\'macro_auc\'] - measured[0][\'macro_auc\']) if eligible else None,\n                          ci_low=interval[0], ci_high=interval[1], bootstrap_requested=bootstrap,\n                          bootstrap_valid=len(deltas), bootstrap_discarded=bootstrap - len(deltas)))\n    return pairs\n\n\ndef compare_runs(run_dirs, out_dir, bootstrap=200, seed=42) -> Path:\n    """Write CSV reports, an equal-probability ensemble, and comparison.json.\n\n    Requires at least two distinct runs with identical validation/cohort IDs,\n    active targets, UIDs and aligned labels, weights, groups and folds. Training\n    supervision/cache IDs may differ and are retained in the summary. All inputs\n    are checked before output creation. Returns the output directory.\n\n    Correlations are Pearson correlations on all validation rows, separately for\n    each active target. Inactive/constant vectors produce None (blank in CSV).\n    AUC uses positive-weight binary labels and a fixed eligible target set.\n    Intervals are paired group-bootstrap 95% percentile intervals for each run\n    minus the first run; a draw losing any eligible class is discarded in full.\n    Zero bootstrap draws disables intervals. Ensemble probabilities are the\n    arithmetic mean, with no ranking, fitting, or validation-prior estimation.\n    """\n    for value, name in [(bootstrap, \'bootstrap\'), (seed, \'seed\')]:\n        if isinstance(value, bool) or not isinstance(value, (int, np.integer)) or value < 0:\n            raise ValueError(f\'{name} must be a nonnegative integer\')\n    bootstrap, seed = int(bootstrap), int(seed)\n    loaded = [_load_run(path) for path in run_dirs]\n    if len(loaded) < 2:\n        raise ValueError(\'At least two runs are required\')\n    summaries, frames, inputs = map(list, zip(*loaded))\n    if len({s[\'run_id\'] for s in summaries}) != len(summaries):\n        raise ValueError(\'Duplicate run_id\')\n    reference = frames[0]\n    for summary, frame in zip(summaries[1:], frames[1:]):\n        for key in [\'validation_id\', \'cohort_id\', \'active_targets\']:\n            if summary[key] != summaries[0][key]:\n                raise ValueError(f"{summary[\'run_id\']}: {key} mismatch")\n        if not reference.index.equals(frame.index):\n            raise ValueError(f"{summary[\'run_id\']}: validation UIDs mismatch")\n        for key in [\'group\', \'fold\'] + YC + WC:\n            if not np.array_equal(reference[key].to_numpy(), frame[key].to_numpy()):\n                raise ValueError(f"{summary[\'run_id\']}: aligned {key} mismatch")\n\n    y, w = (reference[columns].to_numpy(dtype=float) for columns in [YC, WC])\n    active = np.asarray(summaries[0][\'active_targets\'], dtype=bool)\n    eligible = [int(j) for j in np.flatnonzero(active)\n                if np.unique(y[(w[:, j] > 0) & np.isin(y[:, j], [0, 1]), j]).size == 2]\n    predictions = [frame[PC].to_numpy(dtype=float) for frame in frames]\n    measured = [_metrics(y, w, p, active, eligible) for p in predictions]\n    rows = []\n    for summary, metrics in zip(summaries, measured):\n        row = {key: summary[key] for key in SUMMARY_KEYS}\n        row.update({key: summary[\'metrics\'][key] for key in METRICS})\n        row.update(reported_macro_bce=summary[\'metrics\'][\'macro_bce\'],\n                   reported_macro_auc=summary[\'metrics\'][\'macro_auc\'])\n        row.update(metrics)\n        rows.append(row)\n    correlations = []\n    for a, b in combinations(range(len(predictions)), 2):\n        for j, target in enumerate(TARGETS):\n            x, z = predictions[a][:, j], predictions[b][:, j]\n            reason = \'inactive_target\' if not active[j] else (\n                \'constant_predictions\' if np.ptp(x) == 0 or np.ptp(z) == 0 else None)\n            correlation = None\n            if reason is None:\n                # Scaling prevents underflow for very small but nonconstant probabilities.\n                correlation = float(np.corrcoef((x - x.min()) / np.ptp(x),\n                                               (z - z.min()) / np.ptp(z))[0, 1])\n            correlations.append(dict(run_a=summaries[a][\'run_id\'], run_b=summaries[b][\'run_id\'],\n                                     target_index=j, target=target, n_rows=len(x),\n                                     pearson_r=correlation, reason=reason))\n    pairs = _bootstrap_pairs(y, w, predictions, reference[\'group\'].to_numpy(), eligible,\n                             summaries, measured, bootstrap, seed)\n    ensemble_p = np.mean(predictions, axis=0)\n    ensemble = {\'method\': \'equal_probability_mean\', \'exploratory_only\': True,\n                \'run_ids\': [s[\'run_id\'] for s in summaries],\n                \'metrics\': _metrics(y, w, ensemble_p, active, eligible)}\n    identity_payload = dict(schema_version=1, kind=\'experiment_comparison\', inputs=inputs,\n                            bootstrap=bootstrap, seed=seed, role=ROLE)\n    identity = fingerprint(identity_payload)\n    receipt = dict(identity_payload, comparison_id=identity, exploratory_only=True,\n                   validation_id=summaries[0][\'validation_id\'], cohort_id=summaries[0][\'cohort_id\'],\n                   active_targets=active.tolist(), eligible_auc_targets=[TARGETS[j] for j in eligible],\n                   validation_rows=len(reference), validation_groups=reference[\'group\'].nunique(),\n                   summary=rows, correlations=correlations, pairwise=pairs, ensemble=ensemble,\n                   methodology={\n                       \'delta_direction\': \'run_minus_first_run\', \'confidence_level\': .95,\n                       \'interval\': \'paired_group_bootstrap_percentile\',\n                       \'degenerate_replicates\': \'discard_entire_macro_replicate_no_class_filling\',\n                       \'auc\': \'unweighted_ROC_on_positive_weight_binary_labels_fixed_active_targets\',\n                       \'bce\': \'mean_of_per_active_target_weighted_BCE_including_soft_labels\',\n                       \'correlation\': \'per_active_target_Pearson_all_validation_rows\',\n                       \'summary_metrics\': \'BCE_and_AUC_recomputed_reported_values_retained\',\n                       \'constant_baselines\': \'reported_per_run_training_baselines_no_ensemble_baseline\',\n                       \'claims\': \'report_label_exploration_only_no_tuning_or_leaderboard_claim\',\n                   })\n    # Validate serializability before claiming an output directory.\n    fingerprint(receipt)\n    out = safe_output(out_dir, identity)\n    summary_csv = pd.DataFrame(rows)\n    summary_csv[\'eligible_auc_targets\'] = summary_csv[\'eligible_auc_targets\'].map(json.dumps)\n    summary_csv.to_csv(out / \'summary.csv\', index=False)\n    pd.DataFrame(pairs, columns=PAIR_COLUMNS).to_csv(out / \'pairwise.csv\', index=False)\n    pd.DataFrame(correlations, columns=CORRELATION_COLUMNS).to_csv(out / \'pairwise_correlations.csv\', index=False)\n    ensemble_frame = reference[[\'group\', \'fold\'] + YC + WC].copy()\n    ensemble_frame[PC] = ensemble_p\n    ensemble_frame.reset_index().to_csv(out / \'ensemble_predictions.csv\', index=False)\n    receipt[\'outputs\'] = {name: sha256(out / name) for name in\n                          [\'summary.csv\', \'pairwise.csv\', \'pairwise_correlations.csv\', \'ensemble_predictions.csv\']}\n    write_json(out / \'comparison.json\', receipt)\n    return out\n'}
for name, source in SOURCES.items(): (PACKAGE/name).write_text(source, encoding='utf-8')
for name in list(sys.modules):
    if name == 'rsna_experiments' or name.startswith('rsna_experiments.'): del sys.modules[name]
sys.path.insert(0, str(RUNTIME)); importlib.invalidate_caches()
from rsna_experiments.common import *
import pandas as pd, numpy as np
print('Experiment runtime 554732cb8ca9a833')


In [ ]:
FRESH_MANIFEST_PATH = None  # copy the exact Kaggle input path if several copies exist
from rsna_experiments.preparation import prepare
PREP = prepare({'manifest_path': str(locate('fresh_manifest.csv', override=FRESH_MANIFEST_PATH)),
                'out_dir': '/kaggle/working/rsna_full_fold1_preparation',
                'fold': 1, 'max_train': 0, 'max_val': 0, 'seed': 42})
meta = json.loads((PREP/'preparation_meta.json').read_text())
assert meta['selection']['max_train'] == meta['selection']['max_val'] == 0
assert meta['fold'] == 1
assert meta['train_rows'] + meta['validation_rows'] == meta['source_rows']
coverage = pd.DataFrame(meta['target_fold_counts'])
validation_coverage = coverage[coverage.fold.eq(1)].copy()
display(validation_coverage[['target','positive','negative','unknown']])
if (validation_coverage[['positive','negative']] == 0).any().any():
    raise ValueError('A source validation target lacks a positive or negative; review labels/fold before training.')
print('Sparse targets (<10 examples of either class):',
      validation_coverage.loc[validation_coverage[['positive','negative']].min(axis=1).lt(10),
                              'target'].tolist())
print('Studies:', meta['train_rows'], 'train,', meta['validation_rows'], 'validation')
print('Grouping:', meta['grouping_method'], 'Warnings:', meta['warnings'])
print('Save this output:', PREP)


The labels are report-derived. In the current source audit, fold 1 has only two MCL negatives and two synovitis negatives. AUC is defined but highly unstable for those targets. Review target/status samples and grouping before interpreting scores.